# Import the Modules and Load the data

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# Load data
train = pd.read_csv('D:\\Work\\Learnings\\Hackathon\\dataset\\train.csv')
test = pd.read_csv('D:\\Work\\Learnings\\Hackathon\\dataset\\test.csv')

In [9]:
# Select only numeric columns (float or int)
numeric_cols = train.select_dtypes(include=['float64', 'float32', 'int64', 'int32'])

# Drop ID if it's just an identifier
numeric_cols = numeric_cols.drop(columns=['id'], errors='ignore')

# Compute correlation with 'efficiency'
eff_corr = numeric_cols.corr()['efficiency'].drop('efficiency').sort_values(key=abs, ascending=False)

# Show result
print("Correlation of features with efficiency:")
print(eff_corr)


Correlation of features with efficiency:
irradiance            0.580167
soiling_ratio         0.293931
current               0.270045
panel_age            -0.187855
voltage               0.155419
module_temperature   -0.049686
temperature          -0.043876
maintenance_count     0.016274
cloud_coverage       -0.010862
Name: efficiency, dtype: float64


## Data Cleaning Start
- Worked on on float values.

In [10]:
# 1. humidity contain (error, unknow and other strings as well.)
for col in ['humidity', 'wind_speed', 'pressure']: 
    # Step 1: Convert 'humidity' column to float
    train[col] = pd.to_numeric(train[col], errors='coerce')
    test[col] = pd.to_numeric(train[col], errors='coerce')

    # # Step 2: Round the values to 5 decimal places
    # train[col] = train[col].round(5)
    # test[col] = test[col].round(5)

- checked Zero and null value

In [3]:
print(test.isnull().sum().sort_values(ascending=False))
train.isnull().sum().sort_values(ascending=False)

error_code            3611
installation_type     2979
irradiance             615
soiling_ratio          610
maintenance_count      609
panel_age              607
current                587
temperature            582
cloud_coverage         582
module_temperature     580
voltage                547
humidity                83
pressure                79
wind_speed              69
id                       0
string_id                0
dtype: int64


error_code            5912
installation_type     5028
maintenance_count     1027
panel_age             1011
soiling_ratio         1010
cloud_coverage        1010
temperature           1001
voltage                993
irradiance             987
module_temperature     978
current                977
pressure               135
humidity               127
wind_speed             119
id                       0
string_id                0
efficiency               0
dtype: int64

In [4]:
zero_counts = (train == 0).sum().sort_values(ascending=False)
print("Number of zero values in each column:")
print(zero_counts)

Number of zero values in each column:
voltage               5158
efficiency             631
temperature            389
maintenance_count      349
module_temperature      20
id                       1
humidity                 0
panel_age                0
irradiance               0
current                  0
soiling_ratio            0
wind_speed               0
cloud_coverage           0
pressure                 0
string_id                0
error_code               0
installation_type        0
dtype: int64


In [5]:
zero_counts = (test == 0).sum().sort_values(ascending=False)
print("Number of zero values in each column:")
print(zero_counts)

Number of zero values in each column:
voltage               3080
maintenance_count      213
temperature            198
module_temperature      17
id                       1
humidity                 0
panel_age                0
irradiance               0
soiling_ratio            0
current                  0
cloud_coverage           0
wind_speed               0
pressure                 0
string_id                0
error_code               0
installation_type        0
dtype: int64


- Drop rows with 0 or missing efficiency 

In [ ]:
# Drop rows with 0 or missing efficiency
print(train.shape)
train = train[train['efficiency'] > 0].copy()
train.shape


(20000, 17)


(19369, 17)

In [7]:
train

,id,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,cloud_coverage,wind_speed,pressure,string_id,error_code,installation_type,efficiency
0,0,7.817315,576.179270,41.243087,32.135501,4.0,0.803199,37.403527,1.963787,13.691147,62.494044,12.824912,1018.866505,A1,NaN,NaN,0.562096
1,1,24.785727,240.003973,1.359648,19.977460,8.0,0.479456,21.843315,0.241473,27.545096,43.851238,12.012044,1025.623854,D4,E00,dual-axis,0.396447
2,2,46.652695,687.612799,91.265368,1.496401,4.0,0.822398,48.222882,4.191800,43.363708,NaN,1.814400,1010.922654,C3,E00,NaN,0.573776
3,3,53.339567,735.141179,96.190955,18.491582,3.0,0.837529,46.295748,0.960567,57.720436,67.361473,8.736259,1021.846663,A1,NaN,dual-axis,0.629009
4,4,5.575374,12.241203,27.495073,30.722697,6.0,0.551833,0.000000,0.898062,6.786263,3.632000,0.522684,1008.555958,B2,E00,fixed,0.341874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,19995,16.868428,NaN,93.530318,14.393967,3.0,0.738911,12.147711,3.005355,26.206810,1.733013,12.594122,1018.374467,B2,E02,tracking,0.664907
19996,19996,53.415061,296.970303,93.985714,25.997012,2.0,0.513061,0.000000,0.532119,65.000000,64.558667,0.976991,1016.081102,D4,E00,fixed,0.354070
19997,19997,2.442727,660.328019,37.968918,32.818396,9.0,0.548602,13.047950,4.075498,11.584869,57.730134,4.750937,1009.684461,D4,NaN,tracking,0.419734
19998,19998,NaN,632.760700,43.014702,19.063517,4.0,NaN,0.000000,1.068906,21.149351,78.123689,11.304158,1006.673875,A1,E00,tracking,0.661963


- Handled categorical value

In [11]:
for df in [test,train]:
    df['error_code'] = df['error_code'].fillna('Missing')
    df['installation_type'] = df['installation_type'].fillna('Unknown')
    


In [12]:
# Encode categorical features
cat_cols = ['string_id', 'error_code', 'installation_type']
for col in cat_cols:
    train[col] = train[col].astype('category').cat.codes
    test[col] = test[col].astype('category').cat.codes
    

In [13]:
train.to_csv(f'train_cleaned_phase_0.csv', index=False)
test.to_csv(f'test_cleaned_phase_0.csv', index=False)


### Data cleaned End (Worked on categorical value and encoded them)
- need to handle null and nan value

## KNN Imputer to predict value for the zero and nan in train ds

In [88]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

df = train.copy()

# Columns to impute (interdependent)
cols_to_impute = ['voltage', 'current', 'irradiance', 'temperature', 'module_temperature']

# # Replace zero with np.nan ONLY in relevant features
df['voltage'] = df['voltage'].replace(0, np.nan)


In [89]:
# Initialize KNNImputer (you can tune n_neighbors=5 or more)
imputer = KNNImputer(n_neighbors=5)

# Fit and transform
df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])


In [90]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df[cols_to_impute]), columns=cols_to_impute)

# Impute on scaled data
df_scaled_imputed = pd.DataFrame(imputer.fit_transform(df_scaled), columns=cols_to_impute)

# Inverse scale to original
df[cols_to_impute] = scaler.inverse_transform(df_scaled_imputed)


In [ ]:
df.to_csv(f'train_after_knnIputer2.csv', index=False)

In [91]:
print(df.isnull().sum().sort_values(ascending=False))
test.isnull().sum().sort_values(ascending=False)


panel_age             987
maintenance_count     985
cloud_coverage        981
soiling_ratio         972
pressure              131
humidity              122
wind_speed            117
id                      0
temperature             0
current                 0
voltage                 0
irradiance              0
module_temperature      0
string_id               0
error_code              0
installation_type       0
efficiency              0
dtype: int64


irradiance            615
soiling_ratio         610
maintenance_count     609
panel_age             607
current               587
temperature           582
cloud_coverage        582
module_temperature    580
voltage               547
humidity               83
pressure               79
wind_speed             69
id                      0
string_id               0
error_code              0
installation_type       0
dtype: int64

In [48]:
test.shape

(12000, 16)

## KNN Imputer to predict value for the zero and nan in test ds

In [46]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

df = test.copy()

# Columns to impute (interdependent)
cols_to_impute = ['voltage', 'current', 'irradiance', 'temperature', 'module_temperature','cloud_coverage','humidity','soiling_ratio','pressure']

# Replace zero with np.nan ONLY in relevant features
df[cols_to_impute] = df[cols_to_impute].replace(0, np.nan)

# Initialize KNNImputer (you can tune n_neighbors=5 or more)
imputer = KNNImputer(n_neighbors=5)

# Fit and transform
df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df[cols_to_impute]), columns=cols_to_impute)

# Impute on scaled data
df_scaled_imputed = pd.DataFrame(imputer.fit_transform(df_scaled), columns=cols_to_impute)

# Inverse scale to original
df[cols_to_impute] = scaler.inverse_transform(df_scaled_imputed)


In [ ]:
df.to_csv(f'test_after_knnIputer.csv', index=False)

In [47]:
print(df.isnull().sum().sort_values(ascending=False))

maintenance_count     609
panel_age             607
wind_speed             69
id                      0
humidity                0
irradiance              0
soiling_ratio           0
temperature             0
voltage                 0
current                 0
module_temperature      0
cloud_coverage          0
pressure                0
string_id               0
error_code              0
installation_type       0
dtype: int64


In [51]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

# Load your data (replace with your actual file path)
df_test = pd.read_csv('test_after_knnIputer.csv')
df_train = pd.read_csv('train_after_knnIputer.csv')
df_train['is_train'] = 1
df_test['is_train'] = 0

df = pd.concat([df_train, df_test], axis=0, ignore_index=True)

# Columns to impute (interdependent)
cols_to_impute = ['maintenance_count','panel_age']

# Replace zero with np.nan ONLY in relevant features
df[cols_to_impute] = df[cols_to_impute].replace(0, np.nan)

# Initialize KNNImputer (you can tune n_neighbors=5 or more)
imputer = KNNImputer(n_neighbors=5)

# Fit and transform
df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df[cols_to_impute]), columns=cols_to_impute)

# Impute on scaled data
df_scaled_imputed = pd.DataFrame(imputer.fit_transform(df_scaled), columns=cols_to_impute)

# Inverse scale to original
df[cols_to_impute] = scaler.inverse_transform(df_scaled_imputed)

In [52]:
df_train_processed = df[df['is_train'] == 1].drop(columns=['is_train'])
df_test_processed = df[df['is_train'] == 0].drop(columns=['is_train'])


In [64]:
df_train_processed.to_csv(f'Full_cleaned_train.csv', index=False)
df_test_processed.to_csv(f'Full_cleaned_test.csv', index=False)


In [56]:
df_test_processed.shape
df_train_processed.shape

(19369, 17)

In [63]:
print(df_test_processed.isnull().sum().sort_values(ascending=False))
df_train_processed.isnull().sum().sort_values(ascending=False)

wind_speed            69
id                     0
irradiance             0
temperature            0
panel_age              0
maintenance_count      0
soiling_ratio          0
humidity               0
voltage                0
current                0
module_temperature     0
cloud_coverage         0
pressure               0
string_id              0
error_code             0
installation_type      0
dtype: int64


wind_speed            117
id                      0
temperature             0
humidity                0
irradiance              0
maintenance_count       0
soiling_ratio           0
voltage                 0
panel_age               0
current                 0
module_temperature      0
cloud_coverage          0
pressure                0
string_id               0
error_code              0
installation_type       0
efficiency              0
dtype: int64

In [62]:
df_test_processed = df_test_processed.drop(columns=['efficiency'])